In [43]:
import pandas as pd
import re
import os
import emoji
from langdetect import detect
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv("Emotion-Classification-in-E-Commerce/data/raw/dataset.csv")
df.head()

,Rating,Review,Product Name,Product Category,Emotion,Data Source,Sentiment
0,5.0,অসাধারণ ফোন।অনেক পছন্দ হয়েছে।একদম অথেনটিক শাও...,Redmi 12C (4/128GB),Smart Phones,Happy,Daraz,Positive
1,5.0,"Phone is good according to my uses, Upgraded f...",Redmi 12C (4/128GB),Smart Phones,Happy,Daraz,Positive
2,5.0,অল্প দামে দারুন একটা স্মার্টফোন 💙,Redmi 12C (4/128GB),Smart Phones,Love,Daraz,Positive
3,5.0,"Super Fast Delivery ,11200 TK te pailam",Redmi 12C (4/128GB),Smart Phones,Happy,Daraz,Positive
4,5.0,Delay Delivery... Good Product.,Redmi 12C (4/128GB),Smart Phones,Happy,Daraz,Positive


In [34]:
df = df.drop_duplicates()

In [35]:
df = df[['Review', 'Emotion']]
df = df.rename(columns={'Review': 'text', 'Emotion': 'label'})

In [36]:
def is_english(text):
    try:
        return detect(text) == 'en'
    except:
        return False

df = df[df['text'].apply(is_english)]

In [37]:
def clean_text(text):
    text = str(text)
    
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

df['text'] = df['text'].apply(clean_text)

In [38]:
df['text'] = df['text'].apply(lambda x: emoji.demojize(x))

In [39]:
df = df[df['text'].str.strip() != ""]

In [40]:
label_mapping = {
    'Happy': 0,
    'Love': 1,
    'Sadness': 2,
    'Fear': 3,
    'Anger': 4
}

df['label'] = df['label'].map(label_mapping)

In [41]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

In [44]:
os.makedirs("data/processed", exist_ok=True)

train_df.to_csv("data/processed/train.csv", index=False)
val_df.to_csv("data/processed/val.csv", index=False)
test_df.to_csv("data/processed/test.csv", index=False)

In [45]:
print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

print(train_df['label'].value_counts())

Train: (24404, 2)
Val: (3051, 2)
Test: (3051, 2)
label
0    14911
1     6847
2     1653
4      673
3      320
Name: count, dtype: int64


In [46]:
df.head()

,text,label
1,phone is good according to my uses upgraded fr...,0
3,super fast delivery tk te pailam,0
4,delay delivery good product,0
5,delay delivery good product,1
7,authentic product,2
